# Pipeline 04Gd: global Tool-Use (G2a, per feature)

Asked **per feature** (same unit as JSON/Vision), but the LLM gets **all** tools and must
*pull* what it needs — push vs. pull. It writes a `[EFFECT] / [IMPORTANCE] /
[RECOMMENDATION]` description. The tools (`utils.GlobalToolBox`) serve the pre-computed
G0/G1 artifacts: target overview, importances (numeric beeswarm), per-feature curve, and
the plots as images. Tool access is **not** restricted to the asked feature — the "brave"
(minimal) retrieval behaviour is a measured outcome. The `call_log` is persisted as
process data (grounding: does it retrieve at all? does it pull rank/beeswarm? how many
calls?). Output: `results/global/tooluse_{model}_{feature}.json`. Resumable.

In [1]:
from __future__ import annotations

import sys, time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from utils import (
    EXPLANATIONS_DIR, RESULTS_DIR, PROMPTS_DIR, GLOBAL_RESULTS_SUBDIR,
    list_global_features, assemble_global_system_prompt,
    GlobalToolBox, GLOBAL_TOOL_DEFINITIONS, run_global_tool_use_loop,
    build_global_record, run_resumable_global_generation,
)
from utils.llm import _get_client, DEFAULT_MODEL, MAX_TOKENS_GENERATION, strip_scratchpad

LOSS_KEY   = 'poisson_log'
MODEL      = DEFAULT_MODEL
MAX_TOKENS = MAX_TOKENS_GENERATION
XAI_MODELS = ['xgb', 'ebm']

PLOTS_DIR = EXPLANATIONS_DIR / 'plots' / 'global'
OUT_DIR   = RESULTS_DIR / GLOBAL_RESULTS_SUBDIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES = list_global_features('ebm', explanations_dir=EXPLANATIONS_DIR)
client   = _get_client()

print(f'LLM model: {MODEL}')
print(f'Tools:     {[t["name"] for t in GLOBAL_TOOL_DEFINITIONS]}')
print(f'Features:  {len(FEATURES)} x {len(XAI_MODELS)} models = {len(FEATURES) * len(XAI_MODELS)} feature-tasks')
print(f'Output:    {OUT_DIR}')

LLM model: claude-sonnet-4-6
Tools:     ['get_target_overview', 'get_feature_importances', 'get_feature_curve', 'get_feature_plot', 'get_beeswarm_plot']
Features:  9 x 2 models = 18 feature-tasks
Output:    /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation-XAI-Stahl-ss26/results/global


In [2]:
# Tool-Use system prompt per model: shared core + tool-use handover, {{MODEL}} filled.
# Cached per model (the tool-loop sets cache_control on the system each round).
SYSTEM = {m: assemble_global_system_prompt('tooluse', m, prompts_dir=PROMPTS_DIR) for m in XAI_MODELS}
print(SYSTEM['ebm'][:500], '\n...')

You are an expert in explainable AI (XAI). You describe, for staff of a bike rental
company with no technical background, how **one single feature** influences the
predicted demand, always for the feature named in the user message.

## DOMAIN CONTEXT

The Capital Bikeshare system in Washington D.C. rents bikes by the hour. A **EBM**
model predicts how many bikes (`cnt`) are rented in a given hour. every statement you
make is about this EBM model. It was trained with Poisson deviance loss, so a
f 
...


In [3]:
# A fresh GlobalToolBox per (model, feature) so each call has its own call_log.
# The client-side tool loop is not batchable (real-time). Resume/persistence via
# utils.run_resumable_global_generation; process data (tool_calls) kept in the record.

def generate_tooluse(model_name, feature, gen_idx):
    toolbox = GlobalToolBox(model_name, explanations_dir=EXPLANATIONS_DIR,
                            plots_dir=PLOTS_DIR, loss_key=LOSS_KEY)
    user = (
        f'Describe the global effect of the feature "{feature}" on hourly bike demand, '
        f'and how important it is relative to the other features. Retrieve whatever you '
        f'need with the tools before answering.'
    )
    t0 = time.time()
    try:
        text, call_log, in_tok, out_tok, stop_reason = run_global_tool_use_loop(
            client, toolbox, user_message=user, system=SYSTEM[model_name],
            model=MODEL, max_tokens=MAX_TOKENS,
        )
    except Exception as e:
        print(f'  [ERROR] {model_name} {feature}: {type(e).__name__}: {e} -> skip')
        return None
    elapsed = time.time() - t0

    record = build_global_record(
        form='tooluse', model_name=model_name, feature=feature,
        explanation=strip_scratchpad(text),
        usage={'input_tokens': in_tok, 'output_tokens': out_tok},
        llm_model=MODEL, loss_key=LOSS_KEY, elapsed_s=round(elapsed, 2),
        include_cache=False,
        extra={'stop_reason': stop_reason, 'n_tool_calls': len(call_log), 'tool_calls': call_log},
    )
    tools_used = [c['tool'] for c in call_log]
    print(f"  {model_name.upper()} {feature:11} calls={len(call_log)} {tools_used} "
          f"stop={stop_reason} in={in_tok} out={out_tok} t={elapsed:.1f}s")
    return record


results = run_resumable_global_generation(
    form='tooluse', model_names=XAI_MODELS, features=FEATURES,
    out_dir=OUT_DIR, generate=generate_tooluse,
)

totals = {k: sum(r['usage'].get(k, 0) for r in results) for k in ('input_tokens', 'output_tokens')}
avg_calls = (sum(r['n_tool_calls'] for r in results) / len(results)) if results else 0
print(f"\nTotal: {totals}  avg tool calls/feature: {avg_calls:.2f}  ({len(results)} descriptions)")

    [1] get_feature_importances([]) -> ok
    [1] get_feature_curve(['feature']) -> ok
    [1] get_feature_plot(['feature']) -> ok
  XGB weekday     calls=3 ['get_feature_importances', 'get_feature_curve', 'get_feature_plot'] stop=end_turn in=135456 out=1173 t=28.1s

Total: {'input_tokens': 1246938, 'output_tokens': 15913}  avg tool calls/feature: 3.06  (18 descriptions)
